# FT105A - Trabalho 1: análise ponderada da PNAD Contínua

Notebook complementar ao `pnad_dataviz.ipynb`. Aquele testa seis técnicas sobre uma amostra de 50 mil registros; este trabalha três técnicas sobre a base completa do 2º trimestre de 2026 (521.730 registros, 63 variáveis), com peso amostral. A numeração das seções continua a do outro notebook, que vai de 1 a 6.

Antes de rodar, gere o extrato:

```bash
python 05_prepare_extrato.py
```

O arquivo sai em `../../../data/processed/pnadc_2026q2_extrato.parquet` e não vai para o repositório, porque a pasta é ignorada pelo git.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

# o extrato e gerado por 05_prepare_extrato.py
DATA = Path("../../../data/processed").resolve()


## As três visualizações

As três técnicas abaixo saíram da exploração dos microdados completos (521.730 registros, 63 variáveis, arquivo `../../../data/processed/pnadc_2026q2_extrato.parquet`). A amostra de 50 mil do outro notebook não serve aqui por dois motivos: duas das visualizações precisam de taxas por UF, que só fazem sentido com o peso amostral (`V1028`) sobre a base inteira, e a terceira depende de informação do domicílio (presença de crianças), que a gente reconstrói juntando os moradores pelo identificador `UPA + V1008 + V1014`.

Tudo que é taxa, média ou mediana aqui é ponderado por `V1028`. Os totais fecham com o release do IBGE do 2º tri/2026: desocupação 5,4%, informalidade 37,4%, rendimento médio habitual R$ 3.738.

In [2]:
pn = pd.read_parquet(DATA / "pnadc_2026q2_extrato.parquet")

UF_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}
REGIAO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}
ORDEM_REG = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
# paleta Okabe-Ito, a mesma usada nas figuras da parte 1
COR_REGIAO = {"Norte": "#009E73", "Nordeste": "#D55E00", "Sudeste": "#0072B2", "Sul": "#CC79A7", "Centro-Oeste": "#E69F00"}

pn["uf"] = pn["UF"].map(UF_SIGLA)
pn["regiao"] = pn["UF"].str[0].map(REGIAO)
pn["sexo"] = pn["V2007"].map({1: "Homem", 2: "Mulher"})
# preta e parda somadas em "negra", como o IBGE divulga; amarela e indigena ficam de fora pelo n pequeno
pn["cor"] = pn["V2010"].map({1: "Branca", 2: "Negra", 4: "Negra"})
pn["peso"] = pn["V1028"]
pn["renda"] = pn["VD4019"]                 # rendimento habitual de todos os trabalhos
pn["horas"] = pn["VD4031"].astype(float)   # horas habituais em todos os trabalhos
pn["ocupado"] = pn["VD4002"].eq(1).fillna(False).astype(bool)
pn["na_forca"] = pn["VD4001"].eq(1).fillna(False).astype(bool)

# informalidade na definicao do IBGE: empregado sem carteira (privado ou domestico),
# conta propria e empregador sem CNPJ e trabalhador familiar auxiliar.
# empregado publico sem carteira (VD4009 = 6) nao entra; se entrar a taxa sobe de 37,4% pra 40,8%
pos = pn["VD4009"]
sem_cnpj = pn["V4019"].eq(2).fillna(False)
pn["informal"] = (pn["ocupado"] & (pos.isin([2, 4, 10]) | (pos.isin([8, 9]) & sem_cnpj))).astype(bool)

# crianca de 0 a 5 anos no domicilio: conta os moradores pequenos e devolve pra cada pessoa da casa
pn["dom_id"] = pn["UPA"] + pn["V1008"] + pn["V1014"]
criancas_por_dom = pn.loc[pn["V2009"] <= 5].groupby("dom_id").size()
pn["crianca_0a5"] = (pn["dom_id"].map(criancas_por_dom).fillna(0) > 0).astype(bool)


def pct(mask, w):
    return 100 * w[mask].sum() / w.sum()


def media_pond(v, w):
    ok = v.notna()
    return float(np.average(v[ok], weights=w[ok])) if ok.any() else np.nan


def mediana_pond(v, w):
    ok = v.notna()
    v, w = v[ok].to_numpy(dtype=float), w[ok].to_numpy(dtype=float)
    ordem = np.argsort(v)
    acum = np.cumsum(w[ordem])
    return float(v[ordem][np.searchsorted(acum, acum[-1] / 2)])


def gini_pond(v, w):
    ok = v.notna()
    v, w = v[ok].to_numpy(dtype=float), w[ok].to_numpy(dtype=float)
    ordem = np.argsort(v)
    v, w = v[ordem], w[ordem]
    acum_w = np.cumsum(w)
    acum_vw = np.cumsum(v * w)
    return float(1 - 2 * np.sum((acum_vw - v * w / 2) * w) / (acum_w[-1] * acum_vw[-1]))


# conferencia rapida com o release do IBGE
forca = pn[pn["na_forca"]]
ocup = pn[pn["ocupado"]]
print(f"{len(pn):,} pessoas | populacao estimada {pn['peso'].sum() / 1e6:.1f} mi")
print(f"desocupacao {pct(forca['VD4002'].eq(2), forca['peso']):.1f}% | "
      f"informalidade {pct(ocup['informal'], ocup['peso']):.1f}% | "
      f"renda media R$ {media_pond(ocup['renda'], ocup['peso']):,.0f}")

521,730 pessoas | populacao estimada 213.5 mi
desocupacao 5.4% | informalidade 37.4% | renda media R$ 3,738


### 7. Dispersão com bolhas: a geografia da informalidade

Gráfico de dispersão em que cada UF é um ponto e, além das duas posições, o tamanho e a cor carregam mais duas variáveis. Serve para ver a correlação entre dois indicadores e, ao mesmo tempo, quem pesa mais e a que grupo pertence.

Mapeamento: x = informalidade (% dos ocupados, definição do IBGE); y = renda média do trabalho (o menu troca para mediana ou renda por hora); área = número de ocupados; cor = região; rótulo = sigla da UF. No hover entram Gini, desocupação e % com superior completo.


In [3]:
linhas = []
for uf, g in pn.groupby("uf"):
    o = g[g["ocupado"]]
    f = g[g["na_forca"]]
    linhas.append({
        "uf": uf,
        "regiao": g["regiao"].iloc[0],
        "ocupados_mil": o["peso"].sum() / 1e3,
        "informalidade": pct(o["informal"], o["peso"]),
        "renda_mediana": mediana_pond(o["renda"], o["peso"]),
        "renda_media": media_pond(o["renda"], o["peso"]),
        "renda_hora": media_pond(o["renda"] / (o["horas"] * 4.33).where(o["horas"] > 0), o["peso"]),
        "gini": gini_pond(o["renda"], o["peso"]),
        "desocupacao": pct(f["VD4002"].eq(2), f["peso"]),
        "superior": pct(o["VD3004"].eq(7), o["peso"]),
    })
ufs = pd.DataFrame(linhas)

print("correlacao da informalidade com renda media %.2f, mediana %.2f, renda por hora %.2f" % tuple(
    ufs["informalidade"].corr(ufs[c]) for c in ["renda_media", "renda_mediana", "renda_hora"]))
print("UFs com mediana igual ao salario minimo (R$ 1.621):", ", ".join(ufs.loc[ufs["renda_mediana"] == 1621, "uf"]))
ufs.sort_values("informalidade").round(1)

correlacao da informalidade com renda media -0.85, mediana -0.88, renda por hora -0.82
UFs com mediana igual ao salario minimo (R$ 1.621): AL, BA, CE, MA, PA, PB, PE, PI, RN, SE


,uf,regiao,ocupados_mil,informalidade,renda_mediana,renda_media,renda_hora,gini,desocupacao,superior
23,SC,Sul,4550.1,25.4,3000.0,4372.5,25.2,0.4,2.1,27.1
6,DF,Centro-Oeste,1567.0,28.7,3000.0,6171.2,35.9,0.5,6.5,43.2
11,MS,Centro-Oeste,1444.1,29.2,2500.0,3840.2,22.3,0.4,2.7,25.5
22,RS,Sul,5902.0,29.6,2700.0,4168.9,24.3,0.5,4.2,25.1
25,SP,Sudeste,24553.7,30.1,2700.0,4446.5,25.8,0.5,5.4,29.8
17,PR,Sul,6284.8,30.5,2800.0,4204.4,24.7,0.4,3.1,27.3
12,MT,Centro-Oeste,2075.8,34.1,3000.0,4136.6,23.4,0.4,2.2,24.4
8,GO,Centro-Oeste,3884.1,36.1,2500.0,3946.2,22.2,0.5,4.0,22.9
10,MG,Sudeste,11013.2,36.5,2200.0,3463.0,20.6,0.4,3.8,23.0
18,RJ,Sudeste,8262.5,37.1,2300.0,4158.4,24.9,0.5,7.1,29.0


In [4]:
def fig_bolhas_uf(ufs: pd.DataFrame) -> go.Figure:
    colunas_hover = ["renda_mediana", "renda_media", "renda_hora", "gini", "desocupacao", "superior", "ocupados_mil"]
    fig = go.Figure()
    for reg in ORDEM_REG:
        d = ufs[ufs["regiao"] == reg]
        fig.add_trace(go.Scatter(
            x=d["informalidade"], y=d["renda_media"], name=reg,
            mode="markers+text", text=d["uf"], textposition="top center", textfont=dict(size=11),
            marker=dict(
                size=d["ocupados_mil"], sizemode="area", sizemin=5,
                sizeref=2 * ufs["ocupados_mil"].max() / 60 ** 2,
                color=COR_REGIAO[reg], opacity=0.85, line=dict(width=1, color="white"),
            ),
            customdata=d[colunas_hover].to_numpy(),
            hovertemplate=(
                "<b>%{text}</b> (" + reg + ")"
                "<br>informalidade: %{x:.1f}%"
                "<br>renda mediana: R$ %{customdata[0]:,.0f}"
                "<br>renda média: R$ %{customdata[1]:,.0f}"
                "<br>renda por hora: R$ %{customdata[2]:.1f}"
                "<br>Gini da renda: %{customdata[3]:.2f}"
                "<br>desocupação: %{customdata[4]:.1f}%"
                "<br>superior completo: %{customdata[5]:.1f}%"
                "<br>ocupados: %{customdata[6]:,.0f} mil<extra></extra>"
            ),
        ))

    # menu pra trocar a medida do eixo y sem redesenhar o resto
    medidas = {
        "Renda média (R$/mês)": "renda_media",
        "Renda mediana (R$/mês)": "renda_mediana",
        "Renda média por hora (R$)": "renda_hora",
    }
    titulo = "Informalidade x renda do trabalho nas 27 UFs, 2º tri/2026 (área = ocupados; r = {:.2f})"
    botoes = []
    for rotulo, col in medidas.items():
        ys = [ufs.loc[ufs["regiao"] == reg, col].to_numpy() for reg in ORDEM_REG]
        r_col = ufs["informalidade"].corr(ufs[col])
        botoes.append(dict(label=rotulo, method="update",
                           args=[{"y": ys}, {"yaxis.title.text": rotulo, "title.text": titulo.format(r_col)}]))

    fig.update_layout(
        title=titulo.format(ufs["informalidade"].corr(ufs["renda_media"])),
        xaxis_title="Informalidade (% dos ocupados)", yaxis_title="Renda média (R$/mês)",
        legend_title_text="Região", height=620, margin=dict(t=120),
        updatemenus=[dict(buttons=botoes, direction="down", x=0, xanchor="left", y=1.13, yanchor="top")],
    )
    fig.update_xaxes(ticksuffix="%")
    return fig


fig7 = fig_bolhas_uf(ufs)
fig7.show()

### 8. Mapa de calor em pequenos múltiplos: desocupação por idade, sexo e cor

Cada célula é uma combinação de faixa etária e grupo (sexo x cor) e a cor codifica a taxa de desocupação. Repetindo a mesma matriz para o Brasil e para as cinco regiões dá para comparar padrões entre painéis sem perder a leitura de cada um (pequenos múltiplos). É a versão agregada da matriz de pixels do outro notebook: aqui cada célula é uma taxa, não um indivíduo.

Mapeamento: linhas = faixa etária; colunas = sexo x cor (preta e parda somadas em "negra"); cor = taxa de desocupação em escala sequencial de um matiz só; um painel por região. Células com menos de 30 registros na amostra ficam em branco. O hover traz o n amostral e o tamanho da força de trabalho.


In [5]:
FAIXAS = [14, 18, 25, 30, 40, 50, 60, 200]
ROTULO_FAIXA = ["14-17", "18-24", "25-29", "30-39", "40-49", "50-59", "60+"]
GRUPOS = ["Homem branco", "Homem negro", "Mulher branca", "Mulher negra"]
NOME_GRUPO = {("Homem", "Branca"): "Homem branco", ("Homem", "Negra"): "Homem negro",
              ("Mulher", "Branca"): "Mulher branca", ("Mulher", "Negra"): "Mulher negra"}

forca = pn[pn["na_forca"] & pn["cor"].notna()].copy()
forca["faixa"] = pd.cut(forca["V2009"].astype(int), FAIXAS, right=False, labels=ROTULO_FAIXA)
forca["grupo"] = [NOME_GRUPO[(s, c)] for s, c in zip(forca["sexo"], forca["cor"])]
forca["peso_desocupado"] = forca["peso"] * forca["VD4002"].eq(2).astype(float)


def taxa_por_celula(d: pd.DataFrame) -> pd.DataFrame:
    t = (d.groupby(["faixa", "grupo"], observed=True)
           .agg(peso=("peso", "sum"), peso_desocupado=("peso_desocupado", "sum"), n=("peso", "size"))
           .reset_index())
    t["taxa"] = 100 * t["peso_desocupado"] / t["peso"]
    t.loc[t["n"] < 30, "taxa"] = np.nan   # celula pequena demais pra estimar
    return t


paineis = {"Brasil": taxa_por_celula(forca)}
for reg in ORDEM_REG:
    paineis[reg] = taxa_por_celula(forca[forca["regiao"] == reg])


def fig_calor_desocupacao(paineis: dict) -> go.Figure:
    fig = make_subplots(rows=2, cols=3, subplot_titles=list(paineis), shared_yaxes=True,
                        horizontal_spacing=0.04, vertical_spacing=0.16)
    for i, (nome, t) in enumerate(paineis.items()):
        z = t.pivot(index="faixa", columns="grupo", values="taxa").reindex(index=ROTULO_FAIXA, columns=GRUPOS).astype(float)
        n = t.pivot(index="faixa", columns="grupo", values="n").reindex(index=ROTULO_FAIXA, columns=GRUPOS)
        forca_mil = t.pivot(index="faixa", columns="grupo", values="peso").reindex(index=ROTULO_FAIXA, columns=GRUPOS) / 1e3
        texto = np.where(np.isnan(z.values), "", np.round(z.values, 1).astype(str))
        fig.add_trace(go.Heatmap(
            z=z.values, x=GRUPOS, y=ROTULO_FAIXA, coloraxis="coloraxis", xgap=2, ygap=2,
            text=texto, texttemplate="%{text}", textfont=dict(size=11),
            customdata=np.dstack([n.values, forca_mil.values]),
            hovertemplate=(nome + "<br>%{y} anos, %{x}<br>desocupação: %{z:.1f}%"
                           "<br>força de trabalho: %{customdata[1]:,.0f} mil<br>n amostral: %{customdata[0]}<extra></extra>"),
        ), row=i // 3 + 1, col=i % 3 + 1)

    fig.update_layout(
        title="Taxa de desocupação (%) por faixa etária, sexo e cor, Brasil e regiões, 2º tri/2026",
        coloraxis=dict(colorscale=[[0, "#eef4fc"], [1, "#2a78d6"]], cmin=0, cmax=25,
                       colorbar=dict(title="Desocupação (%)", ticksuffix="%")),
        height=760, margin=dict(t=90),
    )
    fig.update_yaxes(autorange="reversed")   # 14-17 no topo
    fig.update_xaxes(tickangle=-25)
    return fig


fig8 = fig_calor_desocupacao(paineis)
fig8.show()

### 9. Diagrama de Sankey: quem fica fora da força de trabalho entre 25 e 49 anos

Um Sankey mostra como uma população se divide em etapas sucessivas; a largura de cada faixa é proporcional ao número de pessoas. A leitura que interessa está na última divisão: a fatia que vai para "fora da força de trabalho" é, na prática, a taxa de não participação de cada caminho. A cor segue o par sexo x criança pequena em casa, então dá para acompanhar cada grupo do começo ao fim (a versão com etapas agregadas perderia essa informação, por isso o nó de criança é separado por sexo).

Mapeamento: etapas = sexo -> há criança de 0 a 5 anos no domicílio -> condição na semana de referência (ocupado, desocupado, fora da força); largura = pessoas de 25 a 49 anos (mil, ponderado); cor = sexo x criança. O menu filtra por escolaridade, a quarta variável, e o rótulo de cada nó já mostra a taxa de participação do grupo.


In [6]:
adultos = pn[(pn["V2009"] >= 25) & (pn["V2009"] <= 49) & pn["sexo"].notna()].copy()
adultos["condicao"] = np.select(
    [adultos["ocupado"].to_numpy(), adultos["VD4002"].eq(2).fillna(False).to_numpy(dtype=bool)],
    ["Ocupado", "Desocupado"], default="Fora da força de trabalho",
)
adultos["escol"] = pd.cut(adultos["VD3004"].astype(float), [0, 2, 5, 7],
                          labels=["Até fundamental incompleto", "Fundamental completo a médio", "Superior (incl. incompleto)"])

CONDICOES = ["Ocupado", "Desocupado", "Fora da força de trabalho"]
PLURAL = {"Homem": "Homens", "Mulher": "Mulheres"}
COMBOS = [("Homem", True), ("Homem", False), ("Mulher", True), ("Mulher", False)]
COR_COMBO = {("Homem", True): "0,114,178", ("Homem", False): "86,180,233",
             ("Mulher", True): "213,94,0", ("Mulher", False): "230,159,0"}   # Okabe-Ito em rgb


def fluxos_sankey(d: pd.DataFrame):
    """valores dos links (mil pessoas) e rotulos dos nos, na ordem fixa usada pela figura"""
    valores, rotulos = [], []
    for sexo in ["Homem", "Mulher"]:
        g = d[d["sexo"] == sexo]
        rotulos.append(f"{PLURAL[sexo]}<br>participação {pct(g['na_forca'], g['peso']):.0f}%")
    for sexo, com in COMBOS:
        g = d[(d["sexo"] == sexo) & (d["crianca_0a5"] == com)]
        valores.append(g["peso"].sum() / 1e3)
        rotulos.append(f"{'com' if com else 'sem'} criança de 0 a 5 em casa<br>participação {pct(g['na_forca'], g['peso']):.0f}%")
    for sexo, com in COMBOS:
        g = d[(d["sexo"] == sexo) & (d["crianca_0a5"] == com)]
        for cond in CONDICOES:
            valores.append(g.loc[g["condicao"] == cond, "peso"].sum() / 1e3)
    rotulos += CONDICOES
    return valores, rotulos


# nos: 0-1 sexo | 2-5 sexo x crianca | 6-8 condicao
origem = [0, 0, 1, 1] + [2 + i for i in range(4) for _ in CONDICOES]
destino = [2, 3, 4, 5] + [6 + j for _ in range(4) for j in range(len(CONDICOES))]
cor_link = [f"rgba({COR_COMBO[c]},0.55)" for c in COMBOS] + [f"rgba({COR_COMBO[c]},0.45)" for c in COMBOS for _ in CONDICOES]
cor_no = ["#0072B2", "#D55E00"] + [f"rgb({COR_COMBO[c]})" for c in COMBOS] + ["#c2c2c2", "#8f8f8f", "#5c5c5c"]

recortes = {"Todas as escolaridades": adultos}
for nivel in adultos["escol"].cat.categories:
    recortes[nivel] = adultos[adultos["escol"] == nivel]


def fig_sankey_participacao(recortes: dict) -> go.Figure:
    primeiro = next(iter(recortes))
    valores, rotulos = fluxos_sankey(recortes[primeiro])
    fig = go.Figure(go.Sankey(
        arrangement="snap", valueformat=",.0f", valuesuffix=" mil",
        node=dict(label=rotulos, color=cor_no, pad=22, thickness=18, line=dict(width=0)),
        link=dict(source=origem, target=destino, value=valores, color=cor_link,
                  hovertemplate="%{source.label} para %{target.label}<br>%{value}<extra></extra>"),
    ))
    botoes = []
    for nome, d in recortes.items():
        v, l = fluxos_sankey(d)
        botoes.append(dict(label=nome, method="update",
                           args=[{"link.value": [v], "node.label": [l]},
                                 {"title.text": f"Pessoas de 25 a 49 anos: sexo, criança pequena em casa e condição na força de trabalho ({nome.lower()})"}]))
    fig.update_layout(
        title=f"Pessoas de 25 a 49 anos: sexo, criança pequena em casa e condição na força de trabalho ({primeiro.lower()})",
        height=620, margin=dict(t=110, l=20, r=20), font=dict(size=12),
        updatemenus=[dict(buttons=botoes, direction="down", x=0, xanchor="left", y=1.12, yanchor="top")],
    )
    return fig


# tabela de apoio pro texto do relatorio
participacao = (adultos.assign(na_forca_p=adultos["peso"] * adultos["na_forca"])
                .groupby(["escol", "sexo", "crianca_0a5"], observed=True)
                .agg(peso=("peso", "sum"), na_forca_p=("na_forca_p", "sum")))
participacao = (100 * participacao["na_forca_p"] / participacao["peso"]).unstack(["sexo", "crianca_0a5"]).round(1)
participacao.columns = [f"{s}, {'com' if c else 'sem'} criança" for s, c in participacao.columns]
display(participacao)

fig9 = fig_sankey_participacao(recortes)
fig9.show()

,"Homem, sem criança","Homem, com criança","Mulher, sem criança","Mulher, com criança"
escol,,,,
Até fundamental incompleto,78.3,87.8,48.9,39.1
Fundamental completo a médio,91.4,95.0,72.4,55.5
Superior (incl. incompleto),95.0,98.5,89.2,80.2


### O que as três mostram

Na dispersão, informalidade e renda andam em sentidos opostos com força (r = -0,85 com a média, -0,88 com a mediana): Santa Catarina tem 25% de informais e uma das maiores rendas; Maranhão, 57% e a menor. Trocando o eixo para a mediana aparece um piso: em dez UFs do Norte e Nordeste a mediana do ocupado é exatamente o salário mínimo. O Distrito Federal foge da reta, com renda alta e informalidade baixa mas o maior Gini do país, coisa que a bolha sozinha não conta e o hover conta.

No mapa de calor, a desvantagem das mulheres negras aparece em quase toda a tabela, mas não em toda: em 38 das 42 combinações de faixa etária e região com dado suficiente a taxa delas é maior que a dos homens brancos, e em 33 das 41 combinações completas é a maior dos quatro grupos. A idade pesa mais que qualquer outro eixo (adolescentes acima de 20%, adultos de 50 anos ou mais abaixo de 4%), e o contraste maior está entre 25 e 29 anos: 9,4% contra 4,2%, razão de 2,25. As exceções se concentram na faixa de 60 anos ou mais, onde as taxas convergem (2,1% e 2,3% no Brasil, razão 1,09). A ordem interna dos dois grupos do meio, homens negros e mulheres brancas, troca de posição entre painéis; o que se mantém são as pontas.

No Sankey, ter criança de 0 a 5 anos em casa puxa a participação das mulheres para baixo (75,6% sem criança, 61,4% com) e a dos homens para cima (90,0% e 94,7%). O tamanho do efeito depende da escolaridade, mas não de forma linear: a diferença é maior no grupo intermediário, fundamental completo a médio (72,4% contra 55,5%, 16,9 pontos), e fica em 9,8 pontos entre quem não completou o fundamental e 9,0 entre quem chegou ao superior. Trocar o menu de escolaridade e olhar só a largura da faixa que chega em "fora da força de trabalho" resume o achado.